In [ ]:
!pip install -U transformers scikit-learn scipy matplotlib imbalanced-learn nlpaug nltk wordnet scikit-multilearn seaborn torch torchvision optuna seaborn

In [ ]:
print('*********************** Part 1 ***********************')
# 0. Data paths
path = '/data/volume_2/EXIST2023_training.json'
additional_path = '/data/volume_2/train_all_tasks.csv'
#test_path='/notebooks/EXIST2023_test_clean-Copy1.json'
print('00000000000000000000000000000000000000000000000')
    
print('*********************** Part 2 ***********************')
# 1. Install packages
#pip install -U transformers scikit-learn scipy matplotlib imbalanced-learn nlpaug nltk wordnet scikit-multilearn seaborn torch torchvision optuna seaborn

# 2. Import libraries

# General libraries
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

# Sklearn libraries
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, ConfusionMatrixDisplay, 
    label_ranking_average_precision_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

# Skmultilearn
from skmultilearn.model_selection import IterativeStratification

# Imbalanced-learn
from imblearn.over_sampling import RandomOverSampler, SMOTE

# NLTK libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

# Transformers
from transformers import (
    BertTokenizer, BertModel,
    DistilBertModel, BertTokenizerFast,
    XLMRobertaTokenizerFast, 
    GPT2Tokenizer
)

# PyTorch libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split

# Nlpaug libraries
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
from nlpaug.augmenter.char import RandomCharAug
from nlpaug.augmenter.word import SynonymAug, RandomWordAug

torch.cuda.empty_cache()

# 3. GPU Setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print('*********************** Part 3 ***********************')

#Part 3
sample_size=1000
num_labels=1
max_length=256
MAX_LENGTH = 128
num_folds = 2

batch_size_values=[32]
#batch_size_values=[16, 32,64]

# Set parameters
num_epochs = 4
batch_size = 32
learning_rate = 3e-05

# Load, preprocess, and combine data
with open(path) as f:
    data = json.load(f)

# Function to preprocess text
def preprocess_text(text):
    # Lowercase the text
    text = text.lower()
    # Remove URLs, special characters and punctuation
    text = re.sub(r'(https?://\S+|www\.\S+)', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # Tokenize the text
    words = word_tokenize(text)
    
    # Lemmatize words using WordNetLemmatizer
    lemmatizer = WordNetLemmatizer()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    
    return " ".join(lemmatized_words)

# Create a dataframe of tweets and their labels
# Modify the loop that creates the DataFrame of tweets and their labels
tweets = []
for tweet in data:
    if data[tweet]['labels_task1'] != []:
        lang = data[tweet]['lang']
        text = preprocess_text(data[tweet]['tweet'])
        label1 = data[tweet]['labels_task1']  # Store the entire list of labels for Task 1
        label2 = data[tweet]['labels_task2']
        label3 = data[tweet]['labels_task3']
        id = data[tweet]['id_EXIST']
        label4 = 'N/A' # Add NA values for Task 4 and Task 5
        label5 = 'N/A'
        source= 'original'
        extra_info = '_'.join(map(str, [data[tweet]['number_annotators'], data[tweet]['annotators'][0], data[tweet]['gender_annotators'][0], data[tweet]['age_annotators'][0],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][1], data[tweet]['gender_annotators'][1], data[tweet]['age_annotators'][1],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][2], data[tweet]['gender_annotators'][2], data[tweet]['age_annotators'][2],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][3], data[tweet]['gender_annotators'][3], data[tweet]['age_annotators'][3],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][4], data[tweet]['gender_annotators'][4], data[tweet]['age_annotators'][4],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][5], data[tweet]['gender_annotators'][5], data[tweet]['age_annotators'][5]]))
        ...
        tweets.append((id, text, extra_info, label1, label2, label3, label4, label5, lang, source))
df = pd.DataFrame(tweets, columns=['id_EXIST','text', 'extra_info', 'label1', 'label2', 'label3', 'label4', 'label5', 'lang', 'source'])

# Compute the proportion of "YES" labels for each record
def preprocess_labels(labels):
    num_yes = sum([1 for label in labels if label == "YES"])
    proportion_yes = num_yes / len(labels)
    return proportion_yes

# Create a new column 'proportion_yes' in the DataFrame
df['proportion_yes'] = df.apply(lambda row: preprocess_labels(row['label1']), axis=1)  # Pass the label1 list directly to preprocess_labels

# Load the additional English dataset
additional_df = pd.read_csv(additional_path)
additional_df['lang']='en'
additional_df['source']='additional'

# Rename columns to match the original dataset
additional_df = additional_df.rename(columns={'rewire_id': 'id', 'label_sexist': 'label1', 'label_category': 'label4', 'label_vector': 'label5'})

additional_df = additional_df.replace('sexist', 'YES')
additional_df = additional_df.replace('not sexist', 'NO')

#additional_df['proportion_yes'] = additional_df['label1'].apply(lambda x: 1 if x == 'YES' else 0)
additional_df['proportion_yes'] = additional_df['label1'].apply(lambda x: 1 if x == 'sexist' else 0)

# Add missing columns to the additional dataframe
additional_df['label2'] = 'N/A'
additional_df['label3'] = 'N/A'
additional_df['extra_info'] = 'N/A'

additional_df = additional_df.rename(columns={'id': 'id_EXIST'})

# Preprocess the text in the additional dataset
#df['text'] = df['text'].apply(preprocess_text)
additional_df['text'] = additional_df['text'].apply(preprocess_text)


# Concatenate the datasets based on id, text, label 1, and lang, source
df = pd.concat([df, additional_df], ignore_index=True)
df=df.iloc[0:sample_size,:]

print('*********************** Part 4 ***********************')

#Part 4
# Define augment_text function
aug_synonym = naw.SynonymAug(aug_src='wordnet')
aug_swap = naw.RandomWordAug(action="swap")
aug_insert = nac.RandomCharAug(action="insert")

def augment_text(text, num_augmentations=3):
    augmented_texts = []

    for _ in range(num_augmentations):
        # Apply the augmentations
        augmented_text = text
        augmented_text = aug_synonym.augment(augmented_text)
        augmented_text = aug_swap.augment(augmented_text)
        augmented_text = aug_insert.augment(augmented_text)

        # Append the augmented text to the list
        augmented_texts.append(augmented_text)

    return augmented_texts

# Perform data augmentation on training data
train_df_augmented = df.copy()

# Initialize an empty list to store the augmented data
train_data_augmented = []

# Loop through the rows in the original DataFrame
for _, row in train_df_augmented.iterrows():
    text = row["text"]
    
    # Generate the augmented texts
    augmented_texts = augment_text(text)
    
    for augmented_text in augmented_texts:
        # Create a dictionary for each augmented text and add it to the list
        train_data_augmented.append(
            {
                "id_EXIST": row["id_EXIST"],
                "text": augmented_text,
                "label1": row["label1"],
                "label2": row["label2"],
                "label3": row["label3"],
                "label4": row["label4"],
                "label5": row["label5"],
                "extra_info": row["extra_info"],
                "lang": row["lang"],
                "source": row["source"],
            }
        )

# Convert the list of dictionaries into a DataFrame
train_df_augmented = pd.DataFrame(train_data_augmented)

# Shuffle the DataFrame and reset the index
train_df_augmented = train_df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

def majority_voting(row):
    num_yes = sum(label == 'YES' for label in row)
    num_no = sum(label == 'NO' for label in row)
    return 'YES' if num_yes > num_no else 'NO'

# Apply majority voting to each row in the 'label1' column
majority_labels = train_df_augmented['label1'].apply(majority_voting)

# Now you can fit the LabelEncoder on this 1D array-like input
label_encoder = LabelEncoder()
binary_labels = label_encoder.fit_transform(majority_labels)

# Convert these to binary labels (1 for 'YES', 0 for 'NO') based on majority voting
labels_as_binary = [1 if label.count('YES') > label.count('NO') else 0 for label in train_df_augmented['label1']]

print('*********************** Part 5 ***********************')

#Part 5

# Previous parts of your code (data augmentation, majority voting, etc.) remain unchanged...

# Splitting the data
X_train_fold, X_test_fold, y_train_fold, y_test_fold = train_test_split(train_df_augmented, labels_as_binary, test_size=0.2, stratify=labels_as_binary, random_state=42)

# Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')
train_encodings = tokenizer(X_train_fold['text'].astype(str).tolist(), truncation=True, padding='max_length', max_length=256, return_tensors="pt")
test_encodings = tokenizer(X_test_fold['text'].astype(str).tolist(), truncation=True, padding='max_length', max_length=256, return_tensors="pt")

# Custom Dataset
class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Creating the datasets
train_dataset = CustomDataset(train_encodings, y_train_fold)
test_dataset = CustomDataset(test_encodings, y_test_fold)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print('*********************** Part 6 ***********************')

#Part 6
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import torch.optim as optim  

from transformers import XLMRobertaModel

device = torch.device("cuda" if torch.cuda.is_available() else "CPU")

learning_rate = learning_rate  # This seems redundant, make sure you have assigned a specific value to it before this line.
warmup_steps = 200
total_steps = num_epochs * (len(train_df_augmented['id_EXIST']) // 16)


# Define a learning rate scheduler
def create_scheduler(learning_rate, warmup_steps, total_steps):
    def lr_lambda(epoch):
        step = epoch * (len(train_df_augmented['id_EXIST']) // 16)
        if step < warmup_steps:
            return learning_rate * (step / warmup_steps)
        return learning_rate * (0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / (total_steps - warmup_steps))))
    return lr_lambda

# Create a function that builds a BERT model based on the hyperparameters
class CustomBERTModel(nn.Module):
    def __init__(self):
        super(CustomBERTModel, self).__init__()
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-uncased')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        #self.cnn = nn.Conv1d(3, 128, kernel_size=3, padding=1)
        self.cnn = nn.Conv1d(in_channels=2304, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(3*128, num_labels)

    def forward(self, input_ids):
        bert_output = self.bert_model(input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        #cnn_out = self.cnn(concatenated)
        cnn_out = self.cnn(concatenated.permute(0, 2, 1))
        x = self.cnn(x)
        x = self.relu(x)
        pooled = self.pool(cnn_out)
        flat = self.flatten(pooled)
        dropped = self.dropout(flat)
        output = self.fc(dropped)
        return output

model = CustomBERTModel().to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.BCEWithLogitsLoss()

# Now create the learning rate scheduler
scheduler = LambdaLR(optimizer, lr_lambda=create_scheduler(learning_rate, warmup_steps, total_steps))

# Hyperparameter Tuning using Optuna
def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    # adjust optimizer and criterion based on hyperparameters
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # training loop
    model.train()
    for epoch in range(num_epochs):
        for batch in train_loader:
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    # validation loop
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for batch in test_loader:
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(inputs)
            predicted = torch.round(torch.sigmoid(outputs))
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = correct / total
    
    return accuracy

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

# Print the best hyperparameters
print(study.best_params)

# Train with the best hyperparameters
# ...

# Note: Saving, Loading, and Evaluation part remains similar to the Keras way but adapted to PyTorch.

In [ ]:
New code

In [ ]:
print('*********************** Part 1 ***********************')
# 0. Data paths
path = '/data/volume_2/EXIST2023_training.json'
additional_path = '/data/volume_2/train_all_tasks.csv'
#test_path='/notebooks/EXIST2023_test_clean-Copy1.json'
print('00000000000000000000000000000000000000000000000')
    
print('*********************** Part 2 ***********************')
# 1. Install packages
#pip install -U transformers scikit-learn scipy matplotlib imbalanced-learn nlpaug nltk wordnet scikit-multilearn seaborn torch torchvision optuna seaborn

# 2. Import libraries

# General libraries
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

# Sklearn libraries
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, ConfusionMatrixDisplay, 
    label_ranking_average_precision_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

# Skmultilearn
from skmultilearn.model_selection import IterativeStratification

# Imbalanced-learn
from imblearn.over_sampling import RandomOverSampler, SMOTE

# NLTK libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

# Transformers
from transformers import (
    BertTokenizer, BertModel,
    DistilBertModel, BertTokenizerFast,
    XLMRobertaTokenizerFast, 
    GPT2Tokenizer
)

# PyTorch libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split

# Nlpaug libraries
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
from nlpaug.augmenter.char import RandomCharAug
from nlpaug.augmenter.word import SynonymAug, RandomWordAug


torch.cuda.empty_cache()


# 3. GPU Setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print('*********************** Part 3 ***********************')

#Part 3
sample_size=1000
num_labels=1
max_length=256
MAX_LENGTH = 128
num_folds = 2

batch_size_values=[32]
#batch_size_values=[16, 32,64]

# Set parameters
num_epochs = 4
batch_size = 32
learning_rate = 3e-05

# Load, preprocess, and combine data
with open(path) as f:
    data = json.load(f)

# Function to preprocess text
def preprocess_text(text):
    # Lowercase the text
    text = text.lower()
    # Remove URLs, special characters and punctuation
    text = re.sub(r'(https?://\S+|www\.\S+)', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # Tokenize the text
    words = word_tokenize(text)
    
    # Lemmatize words using WordNetLemmatizer
    lemmatizer = WordNetLemmatizer()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    
    return " ".join(lemmatized_words)

# Create a dataframe of tweets and their labels
# Modify the loop that creates the DataFrame of tweets and their labels
tweets = []
for tweet in data:
    if data[tweet]['labels_task1'] != []:
        lang = data[tweet]['lang']
        text = preprocess_text(data[tweet]['tweet'])
        label1 = data[tweet]['labels_task1']  # Store the entire list of labels for Task 1
        label2 = data[tweet]['labels_task2']
        label3 = data[tweet]['labels_task3']
        id = data[tweet]['id_EXIST']
        label4 = 'N/A' # Add NA values for Task 4 and Task 5
        label5 = 'N/A'
        source= 'original'
        extra_info = '_'.join(map(str, [data[tweet]['number_annotators'], data[tweet]['annotators'][0], data[tweet]['gender_annotators'][0], data[tweet]['age_annotators'][0],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][1], data[tweet]['gender_annotators'][1], data[tweet]['age_annotators'][1],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][2], data[tweet]['gender_annotators'][2], data[tweet]['age_annotators'][2],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][3], data[tweet]['gender_annotators'][3], data[tweet]['age_annotators'][3],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][4], data[tweet]['gender_annotators'][4], data[tweet]['age_annotators'][4],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][5], data[tweet]['gender_annotators'][5], data[tweet]['age_annotators'][5]]))
        ...
        tweets.append((id, text, extra_info, label1, label2, label3, label4, label5, lang, source))
df = pd.DataFrame(tweets, columns=['id_EXIST','text', 'extra_info', 'label1', 'label2', 'label3', 'label4', 'label5', 'lang', 'source'])

# Compute the proportion of "YES" labels for each record
def preprocess_labels(labels):
    num_yes = sum([1 for label in labels if label == "YES"])
    proportion_yes = num_yes / len(labels)
    return proportion_yes

# Create a new column 'proportion_yes' in the DataFrame
df['proportion_yes'] = df.apply(lambda row: preprocess_labels(row['label1']), axis=1)  # Pass the label1 list directly to preprocess_labels

# Load the additional English dataset
additional_df = pd.read_csv(additional_path)
additional_df['lang']='en'
additional_df['source']='additional'

# Rename columns to match the original dataset
additional_df = additional_df.rename(columns={'rewire_id': 'id', 'label_sexist': 'label1', 'label_category': 'label4', 'label_vector': 'label5'})

additional_df = additional_df.replace('sexist', 'YES')
additional_df = additional_df.replace('not sexist', 'NO')

#additional_df['proportion_yes'] = additional_df['label1'].apply(lambda x: 1 if x == 'YES' else 0)
additional_df['proportion_yes'] = additional_df['label1'].apply(lambda x: 1 if x == 'sexist' else 0)

# Add missing columns to the additional dataframe
additional_df['label2'] = 'N/A'
additional_df['label3'] = 'N/A'
additional_df['extra_info'] = 'N/A'

additional_df = additional_df.rename(columns={'id': 'id_EXIST'})

# Preprocess the text in the additional dataset
#df['text'] = df['text'].apply(preprocess_text)
additional_df['text'] = additional_df['text'].apply(preprocess_text)


# Concatenate the datasets based on id, text, label 1, and lang, source
df = pd.concat([df, additional_df], ignore_index=True)
df=df.iloc[0:sample_size,:]

print('*********************** Part 4 ***********************')

#Part 4
# Define augment_text function
aug_synonym = naw.SynonymAug(aug_src='wordnet')
aug_swap = naw.RandomWordAug(action="swap")
aug_insert = nac.RandomCharAug(action="insert")

def augment_text(text, num_augmentations=3):
    augmented_texts = []

    for _ in range(num_augmentations):
        # Apply the augmentations
        augmented_text = text
        augmented_text = aug_synonym.augment(augmented_text)
        augmented_text = aug_swap.augment(augmented_text)
        augmented_text = aug_insert.augment(augmented_text)

        # Append the augmented text to the list
        augmented_texts.append(augmented_text)

    return augmented_texts

# Perform data augmentation on training data
train_df_augmented = df.copy()

# Initialize an empty list to store the augmented data
train_data_augmented = []

# Loop through the rows in the original DataFrame
for _, row in train_df_augmented.iterrows():
    text = row["text"]
    
    # Generate the augmented texts
    augmented_texts = augment_text(text)
    
    for augmented_text in augmented_texts:
        # Create a dictionary for each augmented text and add it to the list
        train_data_augmented.append(
            {
                "id_EXIST": row["id_EXIST"],
                "text": augmented_text,
                "label1": row["label1"],
                "label2": row["label2"],
                "label3": row["label3"],
                "label4": row["label4"],
                "label5": row["label5"],
                "extra_info": row["extra_info"],
                "lang": row["lang"],
                "source": row["source"],
            }
        )

# Convert the list of dictionaries into a DataFrame
train_df_augmented = pd.DataFrame(train_data_augmented)

# Shuffle the DataFrame and reset the index
train_df_augmented = train_df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

def majority_voting(row):
    num_yes = sum(label == 'YES' for label in row)
    num_no = sum(label == 'NO' for label in row)
    return 'YES' if num_yes > num_no else 'NO'

# Apply majority voting to each row in the 'label1' column
majority_labels = train_df_augmented['label1'].apply(majority_voting)

# Now you can fit the LabelEncoder on this 1D array-like input
label_encoder = LabelEncoder()
binary_labels = label_encoder.fit_transform(majority_labels)

# Convert these to binary labels (1 for 'YES', 0 for 'NO') based on majority voting
labels_as_binary = [1 if label.count('YES') > label.count('NO') else 0 for label in train_df_augmented['label1']]

print('*********************** Part 5 ***********************')

#Part 5

# Previous parts of your code (data augmentation, majority voting, etc.) remain unchanged...

# Splitting the data
X_train_fold, X_test_fold, y_train_fold, y_test_fold = train_test_split(train_df_augmented, labels_as_binary, test_size=0.2, stratify=labels_as_binary, random_state=42)

# Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')
train_encodings = tokenizer(X_train_fold['text'].astype(str).tolist(), truncation=True, padding='max_length', max_length=256, return_tensors="pt")
test_encodings = tokenizer(X_test_fold['text'].astype(str).tolist(), truncation=True, padding='max_length', max_length=256, return_tensors="pt")

# Custom Dataset
class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Creating the datasets
train_dataset = CustomDataset(train_encodings, y_train_fold)
test_dataset = CustomDataset(test_encodings, y_test_fold)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print('*********************** Part 6 ***********************')

#Part 6
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import torch.optim as optim  

from transformers import XLMRobertaModel

device = torch.device("cuda" if torch.cuda.is_available() else "CPU")

learning_rate = learning_rate  # This seems redundant, make sure you have assigned a specific value to it before this line.
warmup_steps = 200
total_steps = num_epochs * (len(train_df_augmented['id_EXIST']) // 16)


# Define a learning rate scheduler
def create_scheduler(learning_rate, warmup_steps, total_steps):
    def lr_lambda(epoch):
        step = epoch * (len(train_df_augmented['id_EXIST']) // 16)
        if step < warmup_steps:
            return learning_rate * (step / warmup_steps)
        return learning_rate * (0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / (total_steps - warmup_steps))))
    return lr_lambda

# Create a function that builds a BERT model based on the hyperparameters

# Train with the best hyperparameters
# ...

# Note: Saving, Loading, and Evaluation part remains similar to the Keras way but adapted to PyTorch.

In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel, XLMRobertaTokenizer, XLMRobertaModel, DistilBertTokenizer, DistilBertModel
from torch.utils.data import DataLoader
import optuna

# Assuming the dataset and the rest of the initial setup remain the same

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')
        
        self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        
        self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        
        # Concatenating the 3 models: bert, xlm_roberta, distilbert
        self.fc = nn.Linear(768 * 3, num_labels)

    def forward(self, input_ids):
        bert_output = self.bert_model(input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(concatenated[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 2

def objective(trial):
    # Hyperparameters 
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 8, 32, log=True)
    num_epochs = 3

    # Model, optimizer, criterion
    model = CustomBERTModel(num_labels=3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    # DataLoader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for batch_idx, batch in enumerate(train_loader):
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            
            # Clear gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # Backward pass
            loss.backward()
            
            # Gradient accumulation
            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()
                
            running_loss += loss.item()
            
        train_loss = running_loss / len(train_loader)

        # Evaluation (you can add this if needed)

        # Clean up
        del inputs, labels, outputs
        torch.cuda.empty_cache()
    
    # For simplicity, return a dummy accuracy (replace with actual accuracy)
    return 0.9

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)

In [ ]:
pip install sentencepiece

In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel, XLMRobertaTokenizer, XLMRobertaModel, DistilBertTokenizer, DistilBertModel
from torch.utils.data import DataLoader
import optuna

# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        
        # Load tokenizers
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
        self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        
        # Load models
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        
        # Concatenating the 3 models: bert, xlm_roberta, distilbert
        self.fc = nn.Linear(768 * 3, num_labels)

    def forward(self, bert_input_ids, xlm_roberta_input_ids, distilbert_input_ids):
        bert_output = self.bert_model(bert_input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(xlm_roberta_input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(distilbert_input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(concatenated[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 4

def objective(trial):
    # Hyperparameters 
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 4, 16, log=True)
    num_epochs = 3

    # Model, optimizer, criterion
    model = CustomBERTModel(num_labels=3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    # DataLoader modification: Tokenize separately for each model
    # Assuming that your dataset returns text, you would tokenize that text for each model
    # Here's a pseudo-code example. You'd have to adjust according to your dataset structure.
    # train_texts = [item['text'] for item in train_dataset]
    # bert_input_ids = [self.bert_tokenizer.encode(text, return_tensors="pt").squeeze(0) for text in train_texts]
    # ... (similarly for other models)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for batch_idx, batch in enumerate(train_loader):
            bert_inputs = batch['bert_input_ids'].to(device)
            xlm_roberta_inputs = batch['xlm_roberta_input_ids'].to(device)
            distilbert_inputs = batch['distilbert_input_ids'].to(device)
            labels = batch['labels'].to(device)
            
            # Clear gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(bert_inputs, xlm_roberta_inputs, distilbert_inputs)
            loss = criterion(outputs, labels)
            
            # Gradient accumulation
            (loss / GRADIENT_ACCUMULATION_STEPS).backward()
            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()
                
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # Clean up
        del bert_inputs, xlm_roberta_inputs, distilbert_inputs, labels, outputs
        torch.cuda.empty_cache()

    return 0.9  # For now, still using a static accuracy

# Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)


In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel, XLMRobertaTokenizer, XLMRobertaModel, DistilBertTokenizer, DistilBertModel
from torch.utils.data import DataLoader
import optuna

# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        
        # Load tokenizers and models with smaller versions
        self.bert_tokenizer = BertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        self.bert_model = BertModel.from_pretrained('distilbert-base-multilingual-cased')
        
        self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        
        self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        
        # Concatenating the 3 models: bert, xlm_roberta, distilbert
        self.fc = nn.Linear(768 * 3, num_labels)

    def forward(self, bert_input_ids, xlm_roberta_input_ids, distilbert_input_ids):
        bert_output = self.bert_model(bert_input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(xlm_roberta_input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(distilbert_input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(concatenated[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 4

def objective(trial):
    # Hyperparameters 
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 4, 16, log=True)
    num_epochs = 3

    # Model, optimizer, criterion
    model = CustomBERTModel(num_labels=3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for batch_idx, batch in enumerate(train_loader):
            bert_inputs = batch['bert_input_ids'].to(device)
            xlm_roberta_inputs = batch['xlm_roberta_input_ids'].to(device)
            distilbert_inputs = batch['distilbert_input_ids'].to(device)
            labels = batch['labels'].to(device)
            
            # Clear gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(bert_inputs, xlm_roberta_inputs, distilbert_inputs)
            loss = criterion(outputs, labels)
            
            # Gradient accumulation
            (loss / GRADIENT_ACCUMULATION_STEPS).backward()
            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()
                
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # Clean up
        del bert_inputs, xlm_roberta_inputs, distilbert_inputs, labels, outputs
        torch.cuda.empty_cache()

    return 0.9  # For now, still using a static accuracy

# Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)


In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel, XLMRobertaTokenizer, XLMRobertaModel, DistilBertTokenizer, DistilBertModel
from torch.utils.data import DataLoader
import optuna

# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        
        # Load tokenizers and models with smaller versions
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')
        
        self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        
        self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        
        # Concatenating the 3 models: bert, xlm_roberta, distilbert
        self.fc = nn.Linear(768 * 3, num_labels)

    def forward(self, bert_input_ids, xlm_roberta_input_ids, distilbert_input_ids):
        bert_output = self.bert_model(bert_input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(xlm_roberta_input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(distilbert_input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(concatenated[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 4

def objective(trial):
    # Hyperparameters 
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 2, 8, log=True)  # Smaller batch sizes
    num_epochs = 3

    # Model, optimizer, criterion
    model = CustomBERTModel(num_labels=3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for batch_idx, batch in enumerate(train_loader):
            bert_inputs = batch['bert_input_ids'].to(device)
            xlm_roberta_inputs = batch['xlm_roberta_input_ids'].to(device)
            distilbert_inputs = batch['distilbert_input_ids'].to(device)
            labels = batch['labels'].to(device)
            
            # Clear gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(bert_inputs, xlm_roberta_inputs, distilbert_inputs)
            loss = criterion(outputs, labels)
            
            # Gradient accumulation
            (loss / GRADIENT_ACCUMULATION_STEPS).backward()
            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()
                
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # Clean up
        del bert_inputs, xlm_roberta_inputs, distilbert_inputs, labels, outputs
        torch.cuda.empty_cache()

    return 0.9  # For now, still using a static accuracy

# Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)


In [ ]:
nvidia-smi

In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel, XLMRobertaTokenizer, XLMRobertaModel, DistilBertTokenizer, DistilBertModel
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from torch.cuda.amp import autocast, GradScaler
import optuna

# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        
        # Load tokenizers and models with smaller versions
        #self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
        #self.bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-uncased')
        
        self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        #self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-small')
        #self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-small')
        
        self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        
        # Concatenating the 3 models: bert, xlm_roberta, distilbert
        self.fc = nn.Linear(768 * 3, num_labels)

    def forward(self, bert_input_ids, xlm_roberta_input_ids, distilbert_input_ids):
        bert_output = self.bert_model(bert_input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(xlm_roberta_input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(distilbert_input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(concatenated[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 4
GRADIENT_CLIP_VALUE = 1.0  # Adjust as needed
scaler = GradScaler()

def objective(trial):
    # Hyperparameters 
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 2, 8, log=True)  # Smaller batch sizes
    num_epochs = 3

    # Model, optimizer, criterion
    model = CustomBERTModel(num_labels=3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for batch_idx, batch in enumerate(train_loader):
            bert_inputs = batch['bert_input_ids'].to(device)
            xlm_roberta_inputs = batch['xlm_roberta_input_ids'].to(device)
            distilbert_inputs = batch['distilbert_input_ids'].to(device)
            labels = batch['labels'].to(device)
            
            # Clear gradients
            optimizer.zero_grad()
            
            # Forward pass with Mixed Precision Training
            with autocast():
                outputs = model(bert_inputs, xlm_roberta_inputs, distilbert_inputs)
                loss = criterion(outputs, labels)
            
            # Gradient accumulation
            scaler.scale(loss / GRADIENT_ACCUMULATION_STEPS).backward()
            
            # Gradient clipping
            clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
            
            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # Clean up
        del bert_inputs, xlm_roberta_inputs, distilbert_inputs, labels, outputs
        torch.cuda.empty_cache()

    return 0.9  # For now, still using a static accuracy

# Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)

In [ ]:
import torch

# Assuming you're using a CUDA-enabled GPU
device = torch.device("cuda:0")

#x = torch.randn((10000, 10000), device=device)

print(f"Memory Allocated: {torch.cuda.memory_allocated(device)/1024**2:.2f} MB")
print(f"Memory Reserved: {torch.cuda.memory_reserved(device)/1024**2:.2f} MB")


In [ ]:
print('**************** Part 2 **********************')

In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel, XLMRobertaTokenizer, XLMRobertaModel, DistilBertTokenizer, DistilBertModel
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from torch.cuda.amp import autocast, GradScaler
import optuna

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-uncased')
        
        self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        
        self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        
        self.fc = nn.Linear(768 * 3, num_labels)

    def forward(self, bert_input_ids, xlm_roberta_input_ids, distilbert_input_ids):
        bert_output = self.bert_model(bert_input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(xlm_roberta_input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(distilbert_input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(concatenated[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 4
GRADIENT_CLIP_VALUE = 1.0  
scaler = GradScaler()

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 2, 8, log=True)
    num_epochs = 3

    model = CustomBERTModel(num_labels=3)

    # Multi-GPU support
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    
    model.to("cuda:0")

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            bert_inputs = batch['bert_input_ids'].to("cuda:0")
            xlm_roberta_inputs = batch['xlm_roberta_input_ids'].to("cuda:0")
            distilbert_inputs = batch['distilbert_input_ids'].to("cuda:0")
            labels = batch['labels'].to("cuda:0")

            optimizer.zero_grad()

            with autocast():
                outputs = model(bert_inputs, xlm_roberta_inputs, distilbert_inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()

            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        del bert_inputs, xlm_roberta_inputs, distilbert_inputs, labels, outputs
        torch.cuda.empty_cache()

    return 0.9  # Using a static accuracy for now

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)


In [ ]:
#New New code

In [ ]:
print('*********************** Part 1 ***********************')
# 0. Data paths
path = '/data/volume_2/EXIST2023_training.json'
additional_path = '/data/volume_2/train_all_tasks.csv'
#test_path='/notebooks/EXIST2023_test_clean-Copy1.json'
    
print('*********************** Part 2 ***********************')
# 1. Install packages
#pip install -U transformers scikit-learn scipy matplotlib imbalanced-learn nlpaug nltk wordnet scikit-multilearn seaborn torch torchvision optuna seaborn

# 2. Import libraries

# General libraries
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

# Sklearn libraries
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, ConfusionMatrixDisplay, 
    label_ranking_average_precision_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

# Skmultilearn
from skmultilearn.model_selection import IterativeStratification

# Imbalanced-learn
from imblearn.over_sampling import RandomOverSampler, SMOTE

# NLTK libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

# Transformers
from transformers import (
    BertTokenizer, BertModel,
    DistilBertModel, BertTokenizerFast,
    XLMRobertaTokenizerFast, 
    GPT2Tokenizer
)

# PyTorch libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split

# Nlpaug libraries
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
from nlpaug.augmenter.char import RandomCharAug
from nlpaug.augmenter.word import SynonymAug, RandomWordAug

torch.cuda.empty_cache()

# 3. GPU Setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print('*********************** Part 3 ***********************')

#Part 3
sample_size=1000
num_labels=1
max_length=256
MAX_LENGTH = 128
num_folds = 2

batch_size_values=[32]
#batch_size_values=[16, 32,64]

# Set parameters
num_epochs = 4
batch_size = 32
learning_rate = 3e-05

# Load, preprocess, and combine data
with open(path) as f:
    data = json.load(f)

# Function to preprocess text
def preprocess_text(text):
    # Lowercase the text
    text = text.lower()
    # Remove URLs, special characters and punctuation
    text = re.sub(r'(https?://\S+|www\.\S+)', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # Tokenize the text
    words = word_tokenize(text)
    
    # Lemmatize words using WordNetLemmatizer
    lemmatizer = WordNetLemmatizer()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    
    return " ".join(lemmatized_words)

# Create a dataframe of tweets and their labels
# Modify the loop that creates the DataFrame of tweets and their labels
tweets = []
for tweet in data:
    if data[tweet]['labels_task1'] != []:
        lang = data[tweet]['lang']
        text = preprocess_text(data[tweet]['tweet'])
        label1 = data[tweet]['labels_task1']  # Store the entire list of labels for Task 1
        label2 = data[tweet]['labels_task2']
        label3 = data[tweet]['labels_task3']
        id = data[tweet]['id_EXIST']
        label4 = 'N/A' # Add NA values for Task 4 and Task 5
        label5 = 'N/A'
        source= 'original'
        extra_info = '_'.join(map(str, [data[tweet]['number_annotators'], data[tweet]['annotators'][0], data[tweet]['gender_annotators'][0], data[tweet]['age_annotators'][0],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][1], data[tweet]['gender_annotators'][1], data[tweet]['age_annotators'][1],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][2], data[tweet]['gender_annotators'][2], data[tweet]['age_annotators'][2],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][3], data[tweet]['gender_annotators'][3], data[tweet]['age_annotators'][3],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][4], data[tweet]['gender_annotators'][4], data[tweet]['age_annotators'][4],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][5], data[tweet]['gender_annotators'][5], data[tweet]['age_annotators'][5]]))
        ...
        tweets.append((id, text, extra_info, label1, label2, label3, label4, label5, lang, source))
df = pd.DataFrame(tweets, columns=['id_EXIST','text', 'extra_info', 'label1', 'label2', 'label3', 'label4', 'label5', 'lang', 'source'])

# Compute the proportion of "YES" labels for each record
def preprocess_labels(labels):
    num_yes = sum([1 for label in labels if label == "YES"])
    proportion_yes = num_yes / len(labels)
    return proportion_yes

# Create a new column 'proportion_yes' in the DataFrame
df['proportion_yes'] = df.apply(lambda row: preprocess_labels(row['label1']), axis=1)  # Pass the label1 list directly to preprocess_labels

# Load the additional English dataset
additional_df = pd.read_csv(additional_path)
additional_df['lang']='en'
additional_df['source']='additional'

# Rename columns to match the original dataset
additional_df = additional_df.rename(columns={'rewire_id': 'id', 'label_sexist': 'label1', 'label_category': 'label4', 'label_vector': 'label5'})

additional_df = additional_df.replace('sexist', 'YES')
additional_df = additional_df.replace('not sexist', 'NO')

#additional_df['proportion_yes'] = additional_df['label1'].apply(lambda x: 1 if x == 'YES' else 0)
additional_df['proportion_yes'] = additional_df['label1'].apply(lambda x: 1 if x == 'sexist' else 0)

# Add missing columns to the additional dataframe
additional_df['label2'] = 'N/A'
additional_df['label3'] = 'N/A'
additional_df['extra_info'] = 'N/A'

additional_df = additional_df.rename(columns={'id': 'id_EXIST'})

# Preprocess the text in the additional dataset
#df['text'] = df['text'].apply(preprocess_text)
additional_df['text'] = additional_df['text'].apply(preprocess_text)


# Concatenate the datasets based on id, text, label 1, and lang, source
df = pd.concat([df, additional_df], ignore_index=True)
df=df.iloc[0:sample_size,:]

print('*********************** Part 4 ***********************')

#Part 4
# Define augment_text function
aug_synonym = naw.SynonymAug(aug_src='wordnet')
aug_swap = naw.RandomWordAug(action="swap")
aug_insert = nac.RandomCharAug(action="insert")

def augment_text(text, num_augmentations=3):
    augmented_texts = []

    for _ in range(num_augmentations):
        # Apply the augmentations
        augmented_text = text
        augmented_text = aug_synonym.augment(augmented_text)
        augmented_text = aug_swap.augment(augmented_text)
        augmented_text = aug_insert.augment(augmented_text)

        # Append the augmented text to the list
        augmented_texts.append(augmented_text)

    return augmented_texts

# Perform data augmentation on training data
train_df_augmented = df.copy()

# Initialize an empty list to store the augmented data
train_data_augmented = []

# Loop through the rows in the original DataFrame
for _, row in train_df_augmented.iterrows():
    text = row["text"]
    
    # Generate the augmented texts
    augmented_texts = augment_text(text)
    
    for augmented_text in augmented_texts:
        # Create a dictionary for each augmented text and add it to the list
        train_data_augmented.append(
            {
                "id_EXIST": row["id_EXIST"],
                "text": augmented_text,
                "label1": row["label1"],
                "label2": row["label2"],
                "label3": row["label3"],
                "label4": row["label4"],
                "label5": row["label5"],
                "extra_info": row["extra_info"],
                "lang": row["lang"],
                "source": row["source"],
            }
        )

# Convert the list of dictionaries into a DataFrame
train_df_augmented = pd.DataFrame(train_data_augmented)

# Shuffle the DataFrame and reset the index
train_df_augmented = train_df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

def majority_voting(row):
    num_yes = sum(label == 'YES' for label in row)
    num_no = sum(label == 'NO' for label in row)
    return 'YES' if num_yes > num_no else 'NO'

# Apply majority voting to each row in the 'label1' column
majority_labels = train_df_augmented['label1'].apply(majority_voting)

# Now you can fit the LabelEncoder on this 1D array-like input
label_encoder = LabelEncoder()
binary_labels = label_encoder.fit_transform(majority_labels)

# Convert these to binary labels (1 for 'YES', 0 for 'NO') based on majority voting
labels_as_binary = [1 if label.count('YES') > label.count('NO') else 0 for label in train_df_augmented['label1']]

print('*********************** Part 5 ***********************')

#Part 5

# Previous parts of your code (data augmentation, majority voting, etc.) remain unchanged...

# Splitting the data
X_train_fold, X_test_fold, y_train_fold, y_test_fold = train_test_split(train_df_augmented, labels_as_binary, test_size=0.2, stratify=labels_as_binary, random_state=42)

# Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')
train_encodings = tokenizer(X_train_fold['text'].astype(str).tolist(), truncation=True, padding='max_length', max_length=256, return_tensors="pt")
test_encodings = tokenizer(X_test_fold['text'].astype(str).tolist(), truncation=True, padding='max_length', max_length=256, return_tensors="pt")

# Custom Dataset
class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Creating the datasets
train_dataset = CustomDataset(train_encodings, y_train_fold)
test_dataset = CustomDataset(test_encodings, y_test_fold)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print('*********************** Part 6 ***********************')

#Part 6
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import torch.optim as optim  

from transformers import XLMRobertaModel

device = torch.device("cuda" if torch.cuda.is_available() else "CPU")

learning_rate = learning_rate  # This seems redundant, make sure you have assigned a specific value to it before this line.
warmup_steps = 200
total_steps = num_epochs * (len(train_df_augmented['id_EXIST']) // 16)


# Define a learning rate scheduler
def create_scheduler(learning_rate, warmup_steps, total_steps):
    def lr_lambda(epoch):
        step = epoch * (len(train_df_augmented['id_EXIST']) // 16)
        if step < warmup_sشزteps:
            return learning_rate * (step / warmup_steps)
        return learning_rate * (0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / (total_steps - warmup_steps))))
    return lr_lambda



In [ ]:
# Create a function that builds a BERT model based on the hyperparameters
class CustomBERTModel(nn.Module):
    def __init__(self):
        super(CustomBERTModel, self).__init__()
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-uncased')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        #self.cnn = nn.Conv1d(3, 128, kernel_size=3, padding=1)
        self.cnn = nn.Conv1d(in_channels=2304, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(3*128, num_labels)

    def forward(self, input_ids):
        bert_output = self.bert_model(input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        #cnn_out = self.cnn(concatenated)
        cnn_out = self.cnn(concatenated.permute(0, 2, 1))
        x = self.cnn(x)
        x = self.relu(x)
        pooled = self.pool(cnn_out)
        flat = self.flatten(pooled)
        dropped = self.dropout(flat)
        output = self.fc(dropped)
        return output

model = CustomBERTModel().to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.BCEWithLogitsLoss()

# Now create the learning rate scheduler
scheduler = LambdaLR(optimizer, lr_lambda=create_scheduler(learning_rate, warmup_steps, total_steps))

# Hyperparameter Tuning using Optuna
def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    # adjust optimizer and criterion based on hyperparameters
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # training loop
    model.train()
    for epoch in range(num_epochs):
        for batch in train_loader:
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    # validation loop
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for batch in test_loader:
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(inputs)
            predicted = torch.round(torch.sigmoid(outputs))
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = correct / total
    
    return accuracy

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

# Print the best hyperparameters
print(study.best_params)

# Train with the best hyperparameters
# ...

# Note: Saving, Loading, and Evaluation part remains similar to the Keras way but adapted to PyTorch.

In [ ]:
import torch

# Assuming you're using a CUDA-enabled GPU
#device = torch.device("cuda:0")

#x = torch.randn((10000, 10000), device=device)

print(f"Memory Allocated: {torch.cuda.memory_allocated(device)/1024**2:.2f} MB")
print(f"Memory Reserved: {torch.cuda.memory_reserved(device)/1024**2:.2f} MB")

In [ ]:
print('*********************** Part 6 ***********************')

import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel, XLMRobertaTokenizer, XLMRobertaModel, DistilBertTokenizer, DistilBertModel
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from torch.cuda.amp import autocast, GradScaler
import optuna

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-uncased')
        
        self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        
        self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        
        self.fc = nn.Linear(768 * 3, num_labels)

    def forward(self, bert_input_ids, xlm_roberta_input_ids, distilbert_input_ids):
        bert_output = self.bert_model(bert_input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(xlm_roberta_input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(distilbert_input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(concatenated[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 4
GRADIENT_CLIP_VALUE = 1.0  
scaler = GradScaler()

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 2, 8, log=True)
    num_epochs = 3

    model = CustomBERTModel(num_labels=3)

    # Multi-GPU support
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    
    model.to("cuda:0")

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    
    # DataLoader modification: Tokenize separately for each model
    # Assuming that your dataset returns text, you would tokenize that text for each model
    # Here's a pseudo-code example. You'd have to adjust according to your dataset structure.
    train_texts = [item['text'] for item in train_dataset]
    bert_input_ids = [self.bert_tokenizer.encode(text, return_tensors="pt").squeeze(0) for text in train_texts]
    xlm_roberta_input_ids = [self.xlm_roberta_tokenizer.encode(text, return_tensors="pt").squeeze(0) for text in train_texts]
    distilbert_input_ids = [self.distilbert_tokenize.encode(text, return_tensors="pt").squeeze(0) for text in train_texts]
    # ... (similarly for other models)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            bert_inputs = batch['bert_input_ids'].to("cuda:0")
            xlm_roberta_inputs = batch['xlm_roberta_input_ids'].to("cuda:0")
            distilbert_inputs = batch['distilbert_input_ids'].to("cuda:0")
            labels = batch['labels'].to("cuda:0")

            optimizer.zero_grad()

            with autocast():
                outputs = model(bert_inputs, xlm_roberta_inputs, distilbert_inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()

            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        del bert_inputs, xlm_roberta_inputs, distilbert_inputs, labels, outputs
        torch.cuda.empty_cache()

    return 0.9  # Using a static accuracy for now

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)

In [ ]:
print(train_dataset[0])


In [ ]:
class CustomDataset(Dataset):
    ...
    def __getitem__(self, idx):
        sample = self.data[idx]
        # Ensure the 'text' key exists in the returned dictionary
        return {'text': sample['text'], ...}


In [ ]:
#New New code

In [ ]:
print('*********************** Part 1 ***********************')
# 0. Data paths
path = '/data/volume_2/EXIST2023_training.json'
additional_path = '/data/volume_2/train_all_tasks.csv'
#test_path='/notebooks/EXIST2023_test_clean-Copy1.json'
    
print('*********************** Part 2 ***********************')
# 1. Install packages
#pip install -U transformers scikit-learn scipy matplotlib imbalanced-learn nlpaug nltk wordnet scikit-multilearn seaborn torch torchvision optuna seaborn

# 2. Import libraries

# General libraries
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

# Sklearn libraries
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, ConfusionMatrixDisplay, 
    label_ranking_average_precision_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

# Skmultilearn
from skmultilearn.model_selection import IterativeStratification

# Imbalanced-learn
from imblearn.over_sampling import RandomOverSampler, SMOTE

# NLTK libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

# Transformers
from transformers import (
    BertTokenizer, BertModel,
    DistilBertModel, BertTokenizerFast,
    XLMRobertaTokenizerFast, 
    GPT2Tokenizer
)

# PyTorch libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split

# Nlpaug libraries
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
from nlpaug.augmenter.char import RandomCharAug
from nlpaug.augmenter.word import SynonymAug, RandomWordAug

torch.cuda.empty_cache()

# 3. GPU Setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print('*********************** Part 3 ***********************')

#Part 3
sample_size=1000
num_labels=1
max_length=256
MAX_LENGTH = 128
num_folds = 2

batch_size_values=[32]
#batch_size_values=[16, 32,64]

# Set parameters
num_epochs = 1
batch_size = 8
learning_rate = 3e-05

# Load, preprocess, and combine data
with open(path) as f:
    data = json.load(f)

# Function to preprocess text
def preprocess_text(text):
    # Lowercase the text
    text = text.lower()
    # Remove URLs, special characters and punctuation
    text = re.sub(r'(https?://\S+|www\.\S+)', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # Tokenize the text
    words = word_tokenize(text)
    
    # Lemmatize words using WordNetLemmatizer
    lemmatizer = WordNetLemmatizer()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    
    return " ".join(lemmatized_words)

# Create a dataframe of tweets and their labels
# Modify the loop that creates the DataFrame of tweets and their labels
tweets = []
for tweet in data:
    if data[tweet]['labels_task1'] != []:
        lang = data[tweet]['lang']
        text = preprocess_text(data[tweet]['tweet'])
        label1 = data[tweet]['labels_task1']  # Store the entire list of labels for Task 1
        label2 = data[tweet]['labels_task2']
        label3 = data[tweet]['labels_task3']
        id = data[tweet]['id_EXIST']
        label4 = 'N/A' # Add NA values for Task 4 and Task 5
        label5 = 'N/A'
        source= 'original'
        extra_info = '_'.join(map(str, [data[tweet]['number_annotators'], data[tweet]['annotators'][0], data[tweet]['gender_annotators'][0], data[tweet]['age_annotators'][0],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][1], data[tweet]['gender_annotators'][1], data[tweet]['age_annotators'][1],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][2], data[tweet]['gender_annotators'][2], data[tweet]['age_annotators'][2],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][3], data[tweet]['gender_annotators'][3], data[tweet]['age_annotators'][3],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][4], data[tweet]['gender_annotators'][4], data[tweet]['age_annotators'][4],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][5], data[tweet]['gender_annotators'][5], data[tweet]['age_annotators'][5]]))
        ...
        tweets.append((id, text, extra_info, label1, label2, label3, label4, label5, lang, source))
df = pd.DataFrame(tweets, columns=['id_EXIST','text', 'extra_info', 'label1', 'label2', 'label3', 'label4', 'label5', 'lang', 'source'])

# Compute the proportion of "YES" labels for each record
def preprocess_labels(labels):
    num_yes = sum([1 for label in labels if label == "YES"])
    proportion_yes = num_yes / len(labels)
    return proportion_yes

# Create a new column 'proportion_yes' in the DataFrame
df['proportion_yes'] = df.apply(lambda row: preprocess_labels(row['label1']), axis=1)  # Pass the label1 list directly to preprocess_labels

# Load the additional English dataset
additional_df = pd.read_csv(additional_path)
additional_df['lang']='en'
additional_df['source']='additional'

# Rename columns to match the original dataset
additional_df = additional_df.rename(columns={'rewire_id': 'id', 'label_sexist': 'label1', 'label_category': 'label4', 'label_vector': 'label5'})

additional_df = additional_df.replace('sexist', 'YES')
additional_df = additional_df.replace('not sexist', 'NO')

#additional_df['proportion_yes'] = additional_df['label1'].apply(lambda x: 1 if x == 'YES' else 0)
additional_df['proportion_yes'] = additional_df['label1'].apply(lambda x: 1 if x == 'sexist' else 0)

# Add missing columns to the additional dataframe
additional_df['label2'] = 'N/A'
additional_df['label3'] = 'N/A'
additional_df['extra_info'] = 'N/A'

additional_df = additional_df.rename(columns={'id': 'id_EXIST'})

# Preprocess the text in the additional dataset
#df['text'] = df['text'].apply(preprocess_text)
additional_df['text'] = additional_df['text'].apply(preprocess_text)


# Concatenate the datasets based on id, text, label 1, and lang, source
df = pd.concat([df, additional_df], ignore_index=True)
df=df.iloc[0:sample_size,:]

print('*********************** Part 4 ***********************')

#Part 4
# Define augment_text function
aug_synonym = naw.SynonymAug(aug_src='wordnet')
aug_swap = naw.RandomWordAug(action="swap")
aug_insert = nac.RandomCharAug(action="insert")

def augment_text(text, num_augmentations=3):
    augmented_texts = []

    for _ in range(num_augmentations):
        # Apply the augmentations
        augmented_text = text
        augmented_text = aug_synonym.augment(augmented_text)
        augmented_text = aug_swap.augment(augmented_text)
        augmented_text = aug_insert.augment(augmented_text)

        # Append the augmented text to the list
        augmented_texts.append(augmented_text)

    return augmented_texts

# Perform data augmentation on training data
train_df_augmented = df.copy()

# Initialize an empty list to store the augmented data
train_data_augmented = []

# Loop through the rows in the original DataFrame
for _, row in train_df_augmented.iterrows():
    text = row["text"]
    
    # Generate the augmented texts
    augmented_texts = augment_text(text)
    
    for augmented_text in augmented_texts:
        # Create a dictionary for each augmented text and add it to the list
        train_data_augmented.append(
            {
                "id_EXIST": row["id_EXIST"],
                "text": augmented_text,
                "label1": row["label1"],
                "label2": row["label2"],
                "label3": row["label3"],
                "label4": row["label4"],
                "label5": row["label5"],
                "extra_info": row["extra_info"],
                "lang": row["lang"],
                "source": row["source"],
            }
        )

# Convert the list of dictionaries into a DataFrame
train_df_augmented = pd.DataFrame(train_data_augmented)

# Shuffle the DataFrame and reset the index
train_df_augmented = train_df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

def majority_voting(row):
    num_yes = sum(label == 'YES' for label in row)
    num_no = sum(label == 'NO' for label in row)
    return 'YES' if num_yes > num_no else 'NO'

# Apply majority voting to each row in the 'label1' column
majority_labels = train_df_augmented['label1'].apply(majority_voting)

# Now you can fit the LabelEncoder on this 1D array-like input
label_encoder = LabelEncoder()
binary_labels = label_encoder.fit_transform(majority_labels)

# Convert these to binary labels (1 for 'YES', 0 for 'NO') based on majority voting
labels_as_binary = [1 if label.count('YES') > label.count('NO') else 0 for label in train_df_augmented['label1']]

print('*********************** Part 5 ***********************')

#Part 5

# Previous parts of your code (data augmentation, majority voting, etc.) remain unchanged...

# Splitting the data
X_train_fold, X_test_fold, y_train_fold, y_test_fold = train_test_split(train_df_augmented, labels_as_binary, test_size=0.2, stratify=labels_as_binary, random_state=42)

# Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')
train_encodings = tokenizer(X_train_fold['text'].astype(str).tolist(), truncation=True, padding='max_length', max_length=256, return_tensors="pt")
test_encodings = tokenizer(X_test_fold['text'].astype(str).tolist(), truncation=True, padding='max_length', max_length=256, return_tensors="pt")

# Custom Dataset
class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Creating the datasets
train_dataset = CustomDataset(train_encodings, y_train_fold)
test_dataset = CustomDataset(test_encodings, y_test_fold)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print('*********************** Part 6 ***********************')

#Part 6
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import torch.optim as optim  

from transformers import XLMRobertaModel

device = torch.device("cuda" if torch.cuda.is_available() else "CPU")

learning_rate = learning_rate  # This seems redundant, make sure you have assigned a specific value to it before this line.
warmup_steps = 200
total_steps = num_epochs * (len(train_df_augmented['id_EXIST']) // 16)


# Define a learning rate scheduler
def create_scheduler(learning_rate, warmup_steps, total_steps):
    def lr_lambda(epoch):
        step = epoch * (len(train_df_augmented['id_EXIST']) // 16)
        if step < warmup_steps:
            return learning_rate * (step / warmup_steps)
        return learning_rate * (0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / (total_steps - warmup_steps))))
    return lr_lambda

# Create a function that builds a BERT model based on the hyperparameters
class CustomBERTModel(nn.Module):
    def __init__(self):
        super(CustomBERTModel, self).__init__()
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-uncased')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        #self.cnn = nn.Conv1d(3, 128, kernel_size=3, padding=1)
        self.cnn = nn.Conv1d(in_channels=2304, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(3*128, num_labels)

    def forward(self, input_ids):
        bert_output = self.bert_model(input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        #cnn_out = self.cnn(concatenated)
        cnn_out = self.cnn(concatenated.permute(0, 2, 1))
        x = self.cnn(x)
        x = self.relu(x)
        pooled = self.pool(cnn_out)
        flat = self.flatten(pooled)
        dropped = self.dropout(flat)
        output = self.fc(dropped)
        return output

model = CustomBERTModel().to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.BCEWithLogitsLoss()

# Now create the learning rate scheduler
scheduler = LambdaLR(optimizer, lr_lambda=create_scheduler(learning_rate, warmup_steps, total_steps))

# Hyperparameter Tuning using Optuna
def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    # adjust optimizer and criterion based on hyperparameters
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # training loop
    model.train()
    for epoch in range(num_epochs):
        for batch in train_loader:
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    # validation loop
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for batch in test_loader:
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(inputs)
            predicted = torch.round(torch.sigmoid(outputs))
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = correct / total
    
    return accuracy

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

# Print the best hyperparameters
print(study.best_params)

# Train with the best hyperparameters
# ...

# Note: Saving, Loading, and Evaluation part remains similar to the Keras way but adapted to PyTorch.

In [ ]:
torch.cuda.empty_cache()


In [ ]:
export PYTORCH_CUDA_ALLOC_CONF="max_split_size_mb=500"


In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn

# Assuming these are defined somewhere in your original code
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class CustomBERTModel(nn.Module):
    def __init__(self):
        super(CustomBERTModel, self).__init__()
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-uncased')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        #self.cnn = nn.Conv1d(3, 128, kernel_size=3, padding=1)
        self.cnn = nn.Conv1d(in_channels=2304, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(3*128, num_labels)

    def forward(self, input_ids):
        bert_output = self.bert_model(input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        #cnn_out = self.cnn(concatenated)
        cnn_out = self.cnn(concatenated.permute(0, 2, 1))
        x = self.cnn(x)
        x = self.relu(x)
        pooled = self.pool(cnn_out)
        flat = self.flatten(pooled)
        dropped = self.dropout(flat)
        output = self.fc(dropped)
        return output

#model = CustomBERTModel().to(device)
# Clear GPU cache memory
torch.cuda.empty_cache()

# Transfer model to GPU
model = CustomBERTModel().to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.BCEWithLogitsLoss()

# Gradient Accumulation: Initialize variables
accumulation_steps = 4
optimizer.zero_grad()

# Assuming you have a training loop somewhere, it would look something like this:

for epoch in range(epochs):
    for step, (inputs, labels) in enumerate(train_dataloader):
        
        outputs = model(inputs.to(device))
        loss = criterion(outputs, labels.to(device))
        
        loss = loss / accumulation_steps  # Normalize the loss because we'll sum losses over accumulation_steps
        loss.backward()
        
        if (step + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()


In [ ]:
import os
import torch

# Adjusting PYTORCH_CUDA_ALLOC_CONF
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb=256'

# Your code
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.cuda.empty_cache()

# Transfer model to GPU
model = CustomBERTModel().to(device)


In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel, XLMRobertaTokenizer, XLMRobertaModel, DistilBertTokenizer, DistilBertModel
from torch.utils.data import DataLoader
import optuna

# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        
        # Load tokenizers and models with smaller versions
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')
        
        self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        
        self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        
        # Concatenating the 3 models: bert, xlm_roberta, distilbert
        self.fc = nn.Linear(768 * 3, num_labels)

    def forward(self, bert_input_ids, xlm_roberta_input_ids, distilbert_input_ids):
        bert_output = self.bert_model(bert_input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(xlm_roberta_input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(distilbert_input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(concatenated[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 4

def objective(trial):
    # Hyperparameters 
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 2, 8, log=True)  # Smaller batch sizes
    num_epochs = 3

    # Model, optimizer, criterion
    model = CustomBERTModel(num_labels=3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for batch_idx, batch in enumerate(train_loader):
            bert_inputs = batch['bert_input_ids'].to(device)
            xlm_roberta_inputs = batch['xlm_roberta_input_ids'].to(device)
            distilbert_inputs = batch['distilbert_input_ids'].to(device)
            labels = batch['labels'].to(device)
            
            # Clear gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(bert_inputs, xlm_roberta_inputs, distilbert_inputs)
            loss = criterion(outputs, labels)
            
            # Gradient accumulation
            (loss / GRADIENT_ACCUMULATION_STEPS).backward()
            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()
                
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # Clean up
        del bert_inputs, xlm_roberta_inputs, distilbert_inputs, labels, outputs
        torch.cuda.empty_cache()

    return 0.9  # For now, still using a static accuracy

# Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)


In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel, XLMRobertaTokenizer, XLMRobertaModel, DistilBertTokenizer, DistilBertModel
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from torch.cuda.amp import autocast, GradScaler
import optuna

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-uncased')
        
        self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        
        self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        
        self.fc = nn.Linear(768 * 3, num_labels)

    def forward(self, bert_input_ids, xlm_roberta_input_ids, distilbert_input_ids):
        bert_output = self.bert_model(bert_input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(xlm_roberta_input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(distilbert_input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(concatenated[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 4
GRADIENT_CLIP_VALUE = 1.0  
scaler = GradScaler()

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 2, 8, log=True)
    num_epochs = 3

    model = CustomBERTModel(num_labels=3)

    # Multi-GPU support
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    
    model.to("cuda:0")

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for batch_idx, batch in enumerate(train_loader):
            bert_inputs = batch['bert_input_ids'].to("cuda:0")
            xlm_roberta_inputs = batch['xlm_roberta_input_ids'].to("cuda:0")
            distilbert_inputs = batch['distilbert_input_ids'].to("cuda:0")
            labels = batch['labels'].to("cuda:0")
            
            optimizer.zero_grad()
            
            with autocast():
                outputs = model(bert_inputs, xlm_roberta_inputs, distilbert_inputs)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss / GRADIENT_ACCUMULATION_STEPS).backward()
            clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
            
            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        del bert_inputs, xlm_roberta_inputs, distilbert_inputs, labels, outputs
        torch.cuda.empty_cache()

    return 0.9  # Using a static accuracy for now

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)


In [ ]:
#New New New New

In [3]:
print('*********************** Part 1 ***********************')
# 0. Data paths
path = '/data/volume_2/EXIST2023_training.json'
additional_path = '/data/volume_2/train_all_tasks.csv'
#test_path='/notebooks/EXIST2023_test_clean-Copy1.json'
    
print('*********************** Part 2 ***********************')
# 1. Install packages
#pip install -U transformers scikit-learn scipy matplotlib imbalanced-learn nlpaug nltk wordnet scikit-multilearn seaborn torch torchvision optuna seaborn

# 2. Import libraries

# General libraries
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

# Sklearn libraries
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, ConfusionMatrixDisplay, 
    label_ranking_average_precision_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

# Skmultilearn
from skmultilearn.model_selection import IterativeStratification

# Imbalanced-learn
from imblearn.over_sampling import RandomOverSampler, SMOTE

# NLTK libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

# Transformers
from transformers import (
    BertTokenizer, BertModel,
    DistilBertModel, BertTokenizerFast,
    XLMRobertaTokenizerFast, 
    GPT2Tokenizer
)

# PyTorch libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split

# Nlpaug libraries
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
from nlpaug.augmenter.char import RandomCharAug
from nlpaug.augmenter.word import SynonymAug, RandomWordAug

torch.cuda.empty_cache()

# 3. GPU Setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print('*********************** Part 3 ***********************')

#Part 3
sample_size=1000
num_labels=1
max_length=256
MAX_LENGTH = 128
num_folds = 2

batch_size_values=[32]
#batch_size_values=[16, 32,64]

# Set parameters
num_epochs = 1
batch_size = 8
learning_rate = 3e-05

# Load, preprocess, and combine data
with open(path) as f:
    data = json.load(f)

# Function to preprocess text
def preprocess_text(text):
    # Lowercase the text
    text = text.lower()
    # Remove URLs, special characters and punctuation
    text = re.sub(r'(https?://\S+|www\.\S+)', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # Tokenize the text
    words = word_tokenize(text)
    
    # Lemmatize words using WordNetLemmatizer
    lemmatizer = WordNetLemmatizer()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    
    return " ".join(lemmatized_words)

# Create a dataframe of tweets and their labels
# Modify the loop that creates the DataFrame of tweets and their labels
tweets = []
for tweet in data:
    if data[tweet]['labels_task1'] != []:
        lang = data[tweet]['lang']
        text = preprocess_text(data[tweet]['tweet'])
        label1 = data[tweet]['labels_task1']  # Store the entire list of labels for Task 1
        label2 = data[tweet]['labels_task2']
        label3 = data[tweet]['labels_task3']
        id = data[tweet]['id_EXIST']
        label4 = 'N/A' # Add NA values for Task 4 and Task 5
        label5 = 'N/A'
        source= 'original'
        extra_info = '_'.join(map(str, [data[tweet]['number_annotators'], data[tweet]['annotators'][0], data[tweet]['gender_annotators'][0], data[tweet]['age_annotators'][0],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][1], data[tweet]['gender_annotators'][1], data[tweet]['age_annotators'][1],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][2], data[tweet]['gender_annotators'][2], data[tweet]['age_annotators'][2],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][3], data[tweet]['gender_annotators'][3], data[tweet]['age_annotators'][3],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][4], data[tweet]['gender_annotators'][4], data[tweet]['age_annotators'][4],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][5], data[tweet]['gender_annotators'][5], data[tweet]['age_annotators'][5]]))
        ...
        tweets.append((id, text, extra_info, label1, label2, label3, label4, label5, lang, source))
df = pd.DataFrame(tweets, columns=['id_EXIST','text', 'extra_info', 'label1', 'label2', 'label3', 'label4', 'label5', 'lang', 'source'])

# Compute the proportion of "YES" labels for each record
def preprocess_labels(labels):
    num_yes = sum([1 for label in labels if label == "YES"])
    proportion_yes = num_yes / len(labels)
    return proportion_yes

# Create a new column 'proportion_yes' in the DataFrame
df['proportion_yes'] = df.apply(lambda row: preprocess_labels(row['label1']), axis=1)  # Pass the label1 list directly to preprocess_labels

# Load the additional English dataset
additional_df = pd.read_csv(additional_path)
additional_df['lang']='en'
additional_df['source']='additional'

# Rename columns to match the original dataset
additional_df = additional_df.rename(columns={'rewire_id': 'id', 'label_sexist': 'label1', 'label_category': 'label4', 'label_vector': 'label5'})

additional_df = additional_df.replace('sexist', 'YES')
additional_df = additional_df.replace('not sexist', 'NO')

#additional_df['proportion_yes'] = additional_df['label1'].apply(lambda x: 1 if x == 'YES' else 0)
additional_df['proportion_yes'] = additional_df['label1'].apply(lambda x: 1 if x == 'sexist' else 0)

# Add missing columns to the additional dataframe
additional_df['label2'] = 'N/A'
additional_df['label3'] = 'N/A'
additional_df['extra_info'] = 'N/A'

additional_df = additional_df.rename(columns={'id': 'id_EXIST'})

# Preprocess the text in the additional dataset
#df['text'] = df['text'].apply(preprocess_text)
additional_df['text'] = additional_df['text'].apply(preprocess_text)


# Concatenate the datasets based on id, text, label 1, and lang, source
df = pd.concat([df, additional_df], ignore_index=True)
df=df.iloc[0:sample_size,:]

print('*********************** Part 4 ***********************')

#Part 4
# Define augment_text function
aug_synonym = naw.SynonymAug(aug_src='wordnet')
aug_swap = naw.RandomWordAug(action="swap")
aug_insert = nac.RandomCharAug(action="insert")

def augment_text(text, num_augmentations=3):
    augmented_texts = []

    for _ in range(num_augmentations):
        # Apply the augmentations
        augmented_text = text
        augmented_text = aug_synonym.augment(augmented_text)
        augmented_text = aug_swap.augment(augmented_text)
        augmented_text = aug_insert.augment(augmented_text)

        # Append the augmented text to the list
        augmented_texts.append(augmented_text)

    return augmented_texts

# Perform data augmentation on training data
train_df_augmented = df.copy()

# Initialize an empty list to store the augmented data
train_data_augmented = []

# Loop through the rows in the original DataFrame
for _, row in train_df_augmented.iterrows():
    text = row["text"]
    
    # Generate the augmented texts
    augmented_texts = augment_text(text)
    
    for augmented_text in augmented_texts:
        # Create a dictionary for each augmented text and add it to the list
        train_data_augmented.append(
            {
                "id_EXIST": row["id_EXIST"],
                "text": augmented_text,
                "label1": row["label1"],
                "label2": row["label2"],
                "label3": row["label3"],
                "label4": row["label4"],
                "label5": row["label5"],
                "extra_info": row["extra_info"],
                "lang": row["lang"],
                "source": row["source"],
            }
        )

# Convert the list of dictionaries into a DataFrame
train_df_augmented = pd.DataFrame(train_data_augmented)

# Shuffle the DataFrame and reset the index
train_df_augmented = train_df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

def majority_voting(row):
    num_yes = sum(label == 'YES' for label in row)
    num_no = sum(label == 'NO' for label in row)
    return 'YES' if num_yes > num_no else 'NO'

# Apply majority voting to each row in the 'label1' column
majority_labels = train_df_augmented['label1'].apply(majority_voting)

# Now you can fit the LabelEncoder on this 1D array-like input
label_encoder = LabelEncoder()
binary_labels = label_encoder.fit_transform(majority_labels)

# Convert these to binary labels (1 for 'YES', 0 for 'NO') based on majority voting
labels_as_binary = [1 if label.count('YES') > label.count('NO') else 0 for label in train_df_augmented['label1']]

print('*********************** Part 5 ***********************')

#Part 5

# Previous parts of your code (data augmentation, majority voting, etc.) remain unchanged...

# Splitting the data
X_train_fold, X_test_fold, y_train_fold, y_test_fold = train_test_split(train_df_augmented, labels_as_binary, test_size=0.2, stratify=labels_as_binary, random_state=42)

# Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')
train_encodings = tokenizer(X_train_fold['text'].astype(str).tolist(), truncation=True, padding='max_length', max_length=256, return_tensors="pt")
test_encodings = tokenizer(X_test_fold['text'].astype(str).tolist(), truncation=True, padding='max_length', max_length=256, return_tensors="pt")

# Custom Dataset
class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Creating the datasets
train_dataset = CustomDataset(train_encodings, y_train_fold)
test_dataset = CustomDataset(test_encodings, y_test_fold)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print('*********************** Part 6 ***********************')

#Part 6
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import torch.optim as optim  

from transformers import XLMRobertaModel

device = torch.device("cuda" if torch.cuda.is_available() else "CPU")

learning_rate = learning_rate  # This seems redundant, make sure you have assigned a specific value to it before this line.
warmup_steps = 200
total_steps = num_epochs * (len(train_df_augmented['id_EXIST']) // 16)


# Define a learning rate scheduler
def create_scheduler(learning_rate, warmup_steps, total_steps):
    def lr_lambda(epoch):
        step = epoch * (len(train_df_augmented['id_EXIST']) // 16)
        if step < warmup_steps:
            return learning_rate * (step / warmup_steps)
        return learning_rate * (0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / (total_steps - warmup_steps))))
    return lr_lambda

# Create a function that builds a BERT model based on the hyperparameters
class CustomBERTModel(nn.Module):
    def __init__(self):
        super(CustomBERTModel, self).__init__()
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-uncased')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        #self.cnn = nn.Conv1d(3, 128, kernel_size=3, padding=1)
        self.cnn = nn.Conv1d(in_channels=2304, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(3*128, num_labels)

    def forward(self, input_ids):
        bert_output = self.bert_model(input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        #cnn_out = self.cnn(concatenated)
        cnn_out = self.cnn(concatenated.permute(0, 2, 1))
        x = self.cnn(cnn_out)
        x = self.relu(x)
        pooled = self.pool(cnn_out)
        flat = self.flatten(pooled)
        dropped = self.dropout(flat)
        output = self.fc(dropped)
        return output

model = CustomBERTModel().to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.BCEWithLogitsLoss()

# Now create the learning rate scheduler
scheduler = LambdaLR(optimizer, lr_lambda=create_scheduler(learning_rate, warmup_steps, total_steps))

# Hyperparameter Tuning using Optuna
def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    # adjust optimizer and criterion based on hyperparameters
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # training loop
    model.train()
    for epoch in range(num_epochs):
        for batch in train_loader:
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    # validation loop
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for batch in test_loader:
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(inputs)
            predicted = torch.round(torch.sigmoid(outputs))
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = correct / total
    
    return accuracy

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

# Print the best hyperparameters
print(study.best_params)

# Train with the best hyperparameters
# ...

*********************** Part 1 ***********************
*********************** Part 2 ***********************
Using device: cuda:0
*********************** Part 3 ***********************


[nltk_data] Downloading package wordnet to
[nltk_data]     /home/hmohammadi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/hmohammadi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/hmohammadi/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


*********************** Part 4 ***********************
*********************** Part 5 ***********************
*********************** Part 6 ***********************


[I 2023-08-24 15:01:42,659] A new study created in memory with name: no-name-28659c09-df0e-4ae9-8853-30293a7e1f0d
/tmp/ipykernel_2792/942882953.py:271: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
[W 2023-08-24 15:01:42,863] Trial 0 failed with parameters: {'learning_rate': 6.194280039544473e-05, 'batch_size': 64} because of the following error: OutOfMemoryError('CUDA out of memory. Tried to allocate 48.00 MiB (GPU 0; 22.06 GiB total capacity; 20.57 GiB already allocated; 15.62 MiB free; 20.83 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF').
Traceback (most recent call last):
  File "/home/hmohammadi/.local/lib

OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB (GPU 0; 22.06 GiB total capacity; 20.57 GiB already allocated; 15.62 MiB free; 20.83 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [4]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel, XLMRobertaTokenizer, XLMRobertaModel, DistilBertTokenizer, DistilBertModel
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from torch.cuda.amp import autocast, GradScaler
import optuna

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-uncased')
        
        self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        
        self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        
        self.fc = nn.Linear(768 * 3, num_labels)

    def forward(self, bert_input_ids, xlm_roberta_input_ids, distilbert_input_ids):
        bert_output = self.bert_model(bert_input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(xlm_roberta_input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(distilbert_input_ids).last_hidden_state
        
        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(concatenated[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 4
GRADIENT_CLIP_VALUE = 1.0  
scaler = GradScaler()

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 2, 8, log=True)
    num_epochs = 3

    model = CustomBERTModel(num_labels=3)

    # Multi-GPU support
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    
    model.to("cuda:0")

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for batch_idx, batch in enumerate(train_loader):
            bert_inputs = batch['bert_input_ids'].to("cuda:0")
            xlm_roberta_inputs = batch['xlm_roberta_input_ids'].to("cuda:0")
            distilbert_inputs = batch['distilbert_input_ids'].to("cuda:0")
            labels = batch['labels'].to("cuda:0")
            
            optimizer.zero_grad()
            
            with autocast():
                outputs = model(bert_inputs, xlm_roberta_inputs, distilbert_inputs)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss / GRADIENT_ACCUMULATION_STEPS).backward()
            clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
            
            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        del bert_inputs, xlm_roberta_inputs, distilbert_inputs, labels, outputs
        torch.cuda.empty_cache()

    return 0.9  # Using a static accuracy for now

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)

[I 2023-08-24 15:01:53,777] A new study created in memory with name: no-name-e415602a-6e47-476c-85c9-b7dfb51ecb43
[W 2023-08-24 15:01:58,188] Trial 0 failed with parameters: {'learning_rate': 1.5501494892765595e-05, 'batch_size': 3} because of the following error: OutOfMemoryError('CUDA out of memory. Tried to allocate 312.00 MiB (GPU 0; 22.06 GiB total capacity; 20.55 GiB already allocated; 27.62 MiB free; 20.81 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF').
Traceback (most recent call last):
  File "/home/hmohammadi/.local/lib/python3.8/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_2792/3668908338.py", line 49, in objective
    model.to("cuda:0")
  File "/home/hmohammadi/.local/lib/python3.8/site-packages/torch/nn/modules/module.py", line 1145, in to
    r

Using 2 GPUs!


OutOfMemoryError: CUDA out of memory. Tried to allocate 312.00 MiB (GPU 0; 22.06 GiB total capacity; 20.55 GiB already allocated; 27.62 MiB free; 20.81 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF